## 三表关联 

In [1]:
import pandas as pd 
from pathlib import Path 


In [2]:
project_root = Path.cwd().parent
project_root
data_path = project_root.joinpath('data', 'raw') 
data_path

PosixPath('/Users/hang1002/Documents/实习/找工作/实习项目一_电商订单分析/olist-delivery-risk/data/raw')

### 这里涉及到.ipynb和.py文件读取路径的差异
- `.ipynb`没有`__file__`因而使用`Path.cwd()`到当前`notebook`所在的目录

- `.py`有`__file__`因而使用`Path(__file__)`查到当前的`.py`文件的相对路径，一般用`.resolve()`变成绝对路径，后`.parent`等的操作

In [3]:
orders = pd.read_csv(data_path.joinpath('olist_orders_dataset.csv'))

# 我们这个项目最终需要预测：一笔订单是否会延迟送达，而不是某一条商品明细是否延迟。
# 因此需要用到items的聚合
items = pd.read_csv(project_root.joinpath('data', 'processed', 'order_items_agg.csv')) 

customers = pd.read_csv(data_path.joinpath('olist_customers_dataset.csv')) 



In [4]:
# 为了merge不增加行数，先检查 

print(orders.shape) 
print(items.shape) 
print(customers.shape)

print(orders['order_id'].is_unique) 

print(items['order_id'].is_unique) 

print(customers['customer_id'].is_unique) 
 


(99441, 8)
(98666, 5)
(99441, 5)
True
True
True


- 这里99441-98666=775,有order但是没有order_items

In [5]:
# merge合并 

orders_items = orders.merge(
    items, 
    on='order_id', 
    how='left'
) 
# 若不用how='left', 默认how='inner'
print(orders.shape) 
print(items.shape) 
print(orders_items.shape) 
print(orders_items['order_id'].is_unique) 

(99441, 8)
(98666, 5)
(99441, 12)
True


In [6]:
orders_items.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
item_count                        775
product_amount                    775
freight_amount                    775
total_amount                      775
dtype: int64

In [7]:
orders_items[
    orders_items['item_count'].isna()
]['order_status'].value_counts() 


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

### 未匹配商品明细订单分析

在将订单表 `orders` 与商品明细聚合表 `order_items_agg` 进行左连接后，发现有 **775 个订单没有对应的商品明细信息**。

这些订单的商品相关字段：

- `item_count`
- `product_amount`
- `freight_amount`
- `total_amount`

均为空，说明这些订单不是部分商品信息缺失，而是整个订单没有进入商品明细记录。

进一步分析这些订单的订单状态：

| order_status | 数量 |
| :--- | ---: |
| unavailable | 603 |
| canceled | 164 |
| created | 5 |
| invoiced | 2 |
| shipped | 1 |

#### 结果分析

1. **unavailable 状态（603 条）**

   占未匹配订单的大多数。

   该状态表示商品不可用，订单可能在商品确认阶段失败，没有进入正常的商品履约流程，因此订单表中存在记录，但没有对应商品明细。

2. **canceled 状态（164 条）**

   该类订单已经取消，通常未完成完整交易流程，因此可能没有生成商品明细记录。

3. **created 状态（5 条）**

   该状态表示订单处于创建阶段。少量订单可能已经生成订单记录，但商品明细尚未同步或交易流程未继续。

4. **invoiced 状态（2 条）**

   该状态表示订单已经进入开票阶段。理论上应该存在商品信息，但少量缺失可能由数据同步异常导致。

5. **shipped 状态（1 条）**

   已发货订单通常应包含商品明细，该记录可能属于数据异常情况，但数量极少，对整体分析影响有限。

#### 总结

这 775 个缺失商品信息的订单主要集中在 `unavailable` 和 `canceled` 两类状态，占比超过 98%。

因此，该缺失现象主要由订单未完成正常履约流程导致，而不是随机的数据缺失。

在后续分析中保留这些订单，并继续采用左连接方式关联其他信息，以保证订单级数据完整性。

In [8]:
# 合并customer

print(customers.isna().sum()) 

orders_full = orders_items.merge(
    customers, 
    on='customer_id', 
    how='left'  
)
orders_full.head()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,item_count,product_amount,freight_amount,total_amount,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,29.99,8.72,38.71,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,118.70,22.76,141.46,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,159.90,19.22,179.12,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,45.00,27.20,72.20,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,19.90,8.72,28.62,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


In [9]:
print(orders.shape) 
print(orders_items.shape) 
print(orders_full.shape)

print(orders_full['order_id'].is_unique) 

print(orders_full['customer_city'].isna().sum()) 
print(orders_full['customer_state'].isna().sum()) 

(99441, 8)
(99441, 12)
(99441, 16)
True
0
0


In [10]:
# 保存最终的订单级数据 
processed_path = project_root.joinpath('data', 'processed') 

output_path = processed_path.joinpath('orders_full.csv') 

orders_full.to_csv(output_path, index=False) 


In [11]:
# 最后的merge检查 
print(orders_full.shape) 
print(orders.shape) 
print(orders_full.duplicated().sum()) 
print(orders_full['order_id'].is_unique) 
print(orders_full[['customer_city', 'customer_state']].isna().sum()) 
print(orders_full.columns)

(99441, 16)
(99441, 8)
0
True
customer_city     0
customer_state    0
dtype: int64
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'item_count', 'product_amount', 'freight_amount', 'total_amount',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state'],
      dtype='str')


## 最终订单级数据集检查

经过订单表（`orders`）、商品明细聚合表（`order_items_agg`）以及客户信息表（`customers`）关联后，得到最终订单级数据集 `orders_full`。

该数据集用于后续订单延迟分析和预测建模。

---

### 1. 数据规模

最终数据规模：

| 数据集 | 行数 | 列数 |
| --- | ---: | ---: |
| orders_full | 99441 | 16 |

数据粒度：

> 一行代表一个订单。

经过商品明细聚合后，商品信息已经转换为订单级特征，包括：

- 商品数量（`item_count`）
- 商品总金额（`product_amount`）
- 运费金额（`freight_amount`）
- 订单总金额（`total_amount`）

同时加入客户信息：

- 客户城市（`customer_city`）
- 客户州（`customer_state`）

---

### 2. 订单唯一性检查

检查订单编号是否唯一：

```python
orders_full['order_id'].is_unique